In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
FILES_DIR = '/content/drive/MyDrive/Cassava_Project/ֆայլեր'
if FILES_DIR not in sys.path:
    sys.path.append(FILES_DIR)

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.optim as optim
from train import train_model
import matplotlib.pyplot as plt
from data_loading import load_data
from dataset import get_dataloaders
from augmentations import get_transforms
from preprocesing import prepare_and_save_splits
from dataset import get_dataloaders, get_weighted_criterion
from model import BaselineCNN, DINOv2Cassava, ResNet50Cassava, EfficientNetV2SCassava, unfreeze_last_block
from utils import evaluate_model, verify_transfer_setup, calculate_class_weights, save_history_json, load_history, plot_training_curves, freeze_backbone


import importlib
import evaluate
importlib.reload(evaluate)
from evaluate import run_cassava_evaluation

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
print(" Current working directory:", os.getcwd())
print(" Files here:", os.listdir())

 Current working directory: /content
 Files here: ['.config', 'drive', 'sample_data']


In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/Cassava_Project'
os.chdir(PROJECT_DIR)
print(" Current Working Directory:", os.getcwd())

 Current Working Directory: /content/drive/MyDrive/Cassava_Project


In [ ]:
FILES_DIR = os.path.join(os.getcwd(), 'ֆայլեր')

if FILES_DIR not in sys.path:
    sys.path.append(FILES_DIR)

In [ ]:

# Remove cached custom modules from memory
modules_to_reset = ['utils', 'dataset', 'data_loading', 'preprocesing', 'augmentations']
for module in modules_to_reset:
    if module in sys.modules:
        del sys.modules[module]

print(" Module cache cleared!")

 Module cache cleared!


In [ ]:

dataset_path = '/content/drive/MyDrive/Cassava_Project/cassava_data'
# TRAIN_CSV_PATH = os.path.join(dataset_path, "train.csv")

In [ ]:

# 2. Ներմուծում ենք load_data ֆունկցիան data_loading.py-ից

print("⏳ Executing load_data() from data_loading.py...")
df, mapping, dataset_path = load_data()

print("\n DATA LOADING SUCCESSFUL!")
print(f" Dataset Path: {dataset_path}")
print(f" Total Dataset Rows: {len(df)}")
print("\n DataFrame Preview:")
print(df.head())

⏳ Executing load_data() from data_loading.py...


100%|██████████| 5.76G/5.76G [01:16<00:00, 81.3MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/cassava-leaf-disease-classification

 DATA LOADING SUCCESSFUL!
 Dataset Path: /root/.cache/kagglehub/competitions/cassava-leaf-disease-classification
 Total Dataset Rows: 21397

 DataFrame Preview:
         image_id  label
0  1000015157.jpg      0
1  1000201771.jpg      3
2   100042118.jpg      1
3  1000723321.jpg      1
4  1000812911.jpg      3


In [ ]:
TRAIN_CSV_PATH = os.path.join(dataset_path, "train.csv")

In [ ]:
print("\n⏳ Running prepare_and_save_splits...")

train_df, val_df = prepare_and_save_splits(csv_path=TRAIN_CSV_PATH)


⏳ Running prepare_and_save_splits...
 Train և Validation CSV-ները պահպանվեցին: /content/drive/MyDrive/Cassava_Project
Training set size: 17117 images (80.0%)
Validation set size: 4280 images (20.0%)

--- Class Distribution Comparison ---
       Train (%)  Validation (%)
label                           
0           5.08            5.07
1          10.23           10.23
2          11.15           11.14
3          61.49           61.50
4          12.04           12.06

Overlap between Train and Validation: 0 images


In [ ]:
train_df = pd.read_csv('train_split.csv')
val_df = pd.read_csv('val_split.csv')

print(f" Train samples: {len(train_df)}, Validation samples: {len(val_df)}")

 Train samples: 17117, Validation samples: 4280


In [ ]:
files_path = '/content/drive/MyDrive/Cassava_Project/ֆայլեր'
if files_path not in sys.path:
    sys.path.append(files_path)

print(" Ուղին հաջողությամբ ավելացվեց:")

 Ուղին հաջողությամբ ավելացվեց:


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_DIR = '/content/drive/MyDrive/Cassava_Project/cassava_data/train_images'

train_loader, valid_loader = get_dataloaders(
    train_df=train_df,
    val_df=val_df,
    img_dir=IMG_DIR,
    img_size=384,
    batch_size=16,
    num_workers=4
)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


#BaselineCNN

In [ ]:
checkpoint_path = "/content/drive/MyDrive/Cassava_Project/checkpoints/baseline_cnn.pth"

NUM_CLASSES = 5  # Cassava Leaf Disease classes (0 to 4)
baseline_model = BaselineCNN(num_classes=NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(baseline_model.parameters(), lr=0.0003, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)


if os.path.exists(checkpoint_path):
    print(" Checkpoint-ը գտնվեց։ Բեռնում ենք պահված քաշերը, թրեյնինգ չենք անում...")
    baseline_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    baseline_model.eval()
else:
    print(" Checkpoint-ը չգտնվեց։ Սկսում ենք թրեյնինգը...")
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(baseline_model.parameters(), lr=0.001)

    # Կանչում ենք հենց այս engine.py-ի ֆունկցիան
    history = train_model(
        model=baseline_model,
        train_loader=train_loader,
        val_loader=valid_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=5,
        save_path=checkpoint_path,
        use_amp=True,          # AMP-ը միացված է արագության համար
        monitor="val_acc"      # Կարող ես դնել նաև 'val_loss'
    )

# 3. Վերջում գնահատում ենք մոդելը
evaluate_model(
    model=baseline_model,
    val_loader=valid_loader,
    device=device,
    model_name="Baseline CNN"
)

#DINOv2

In [ ]:
train_transform, val_transform = get_transforms(img_size=224)
CHECKPOINT_DIR="/content/drive/MyDrive/Cassava_Project/checkpoints"


# 5. Դատալոդերների կառուցում (batch_size=32 իդեալական է ViT-ների համար)
train_loader, valid_loader = get_dataloaders(
    train_df=train_df,
    val_df=val_df,
    img_dir=IMG_DIR,
    img_size=224,     # Քո ուզած չափսը DINOv2-ի համար
    batch_size=32,    # ViT/DINOv2-ի համար 32-ը շատ լավ է
    num_workers=2
)

print(" DINOv2 Դատալոդերները պատրաստ են (224x224)!")

# 6. Մոդելի, Loss-ի, Optimizer-ի և Scheduler-ի ինիցիալիզացիա
model_name = "DINOv2 ViT-Base"
model = DINOv2Cassava(num_classes=5, freeze_backbone=False).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

checkpoint_path = os.path.join(CHECKPOINT_DIR, "dinov2_vitb14_best.pth")

# 7. Չեքփոինթի ստուգում և թրեյնինգ
if os.path.exists(checkpoint_path):
    print(f" {model_name}-ի Checkpoint-ը գտնվեց։ Բեռնում ենք պահված քաշերը...")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    print(" Մոդելը հաջողությամբ բեռնվեց:")
else:
    print(f" Checkpoint-ը չգտնվեց։ Սկսում ենք {model_name}-ի վարժեցումը...")
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=valid_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=5,
        scheduler=scheduler,
        save_path=checkpoint_path,
        use_amp=True,
        monitor="val_acc"
    )

# 8. Վերջնական գնահատում
evaluate_model(
    model=model,
    val_loader=valid_loader,
    device=device,
    model_name=model_name
)

# Resnet

In [ ]:
CHECKPOINT_DIR="/content/drive/MyDrive/Cassava_Project/checkpoints"
train_loader, valid_loader = get_dataloaders(
  train_df=train_df,
  val_df=val_df,
  img_dir=IMG_DIR,
  img_size=384,
  batch_size=16, # 384x384 չափսի համար 16-ը հարմար է հիշողության (VRAM) համար
  num_workers=2
)
print(" ResNet50 Դատալոդերները պատրաստ են (384x384)!")

# 5. Մոդելի, Loss-ի, Optimizer-ի և Scheduler-ի ինիցիալիզացիա
model_name = "ResNet50"
model = ResNet50Cassava(num_classes=5, freeze_backbone=False).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

checkpoint_path = os.path.join(CHECKPOINT_DIR, "resnet50_head_best.pth")

if os.path.exists(checkpoint_path):
  print(f" {model_name}-ի Checkpoint-ը գտնվեց։ Բեռնում ենք քաշերը...")
  model.load_state_dict(torch.load(checkpoint_path, map_location=device))
  model.eval()
  print(" Մոդելը հաջողությամբ բեռնվեց:")
else:
  print(f"Checkpoint-ը չգտնվեց։ Սկսում ենք {model_name}-ի վարժեցումը...")
  history = train_model(
  model=model,
  train_loader=train_loader,
  val_loader=valid_loader,
  criterion=criterion,
  optimizer=optimizer,
  device=device,
  epochs=5,
  cheduler=scheduler,
  save_path=checkpoint_path,
  use_amp=True,
  monitor="val_acc"
 )

verify_transfer_setup(model, model_name, device=device)

# 7. Վերջնական գնահատում և մատրիցի պահպանում
evaluate_model(
  model=model,
  val_loader=valid_loader,
  device=device,
  model_name=model_name,
  save_dir="/content/drive/MyDrive/Cassava_Project/plots"
)

# Resnet Fine-tuned

In [ ]:


RESNET_FINETUNED_PATH = '/content/drive/MyDrive/Cassava_Project/checkpoints/resnet50_finetuned_best.pth'
history_dir = '/content/drive/MyDrive/Cassava_Project/history'
finetune_history_path = os.path.join(history_dir, 'resnet50_finetuned_history.json')
full_history_path = os.path.join(history_dir, 'resnet50_full_history.json')

if os.path.exists(RESNET_FINETUNED_PATH):
    print(f"Fine-tuned չեքփոինթը գտնվեց։ Բեռնում ենք քաշերը...")

    checkpoint_state = torch.load(RESNET_FINETUNED_PATH, map_location=device)

    # Ճշգրտում ենք բանալիները (Prefix-ները և վերջին fc շերտը)
    fixed_state_dict = {}
    for k, v in checkpoint_state.items():
        new_key = k if k.startswith("resnet.") else "resnet." + k

        if new_key == "resnet.fc.weight":
            fixed_state_dict["resnet.fc.1.weight"] = v
        elif new_key == "resnet.fc.bias":
            fixed_state_dict["resnet.fc.1.bias"] = v
        else:
            fixed_state_dict[new_key] = v

    model.load_state_dict(fixed_state_dict)
    model.eval()
    print("Fine-tuned ResNet50 մոդելը հաջողությամբ բեռնվեց:")

    if os.path.exists(finetune_history_path):
        history_resnet_finetune = load_history(finetune_history_path)
        print("Fine-tune history-ն հաջողությամբ բեռնվեց ֆայլից:")
    else:
        print("Fine-tune history JSON ֆայլը չգտնվեց:")
        history_resnet_finetune = {}
else:
    print("Fine-tuned չեքփոինթը չգտնվեց։ Սկսում ենք fine-tuning-ը...")

    # 1. Բացում ենք վերջին բլոկը ուտիլիսից
    resnet_model = unfreeze_last_block(model)

    # 2. Ստեղծում ենք ֆայնթյունինգի օպտիմիզատորը
    finetune_optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, resnet_model.parameters()),
        lr=1e-5, weight_decay=1e-4
    )

    # 3. Գործարկում ենք թրեյնինգը
    history_resnet_finetune = train_model(
        model=resnet_model,
        train_loader=train_loader,
        val_loader=valid_loader,
        criterion=criterion,
        optimizer=finetune_optimizer,
        device=device,
        epochs=5,
        save_path=RESNET_FINETUNED_PATH,
        use_amp=True,
        monitor="val_acc"
    )

    # Պահպանում ենք առանձին ֆայնթյունի հիսթրին
    save_history_json(history_resnet_finetune, os.path.join(history_dir, 'resnet50_finetuned_history.json'))

# 4. Վերջում միավորում ենք հեդի և ֆայնթյունի պատմությունները ու պահպանում մեկ ընդհանուր ֆայլում
history_resnet_head = load_history(os.path.join(history_dir, 'resnet50_head_history.json'))

resnet_full_history = {
    "head_only": history_resnet_head,
    "finetuned": history_resnet_finetune
}

save_history_json(resnet_full_history, full_history_path)
print("ResNet50 ամբողջական պատմությունը հաջողությամբ միավորվեց և պահպանվեց:")

In [ ]:


#Loading histories
head_history_path = os.path.join(history_dir, 'resnet50_head_history.json')
finetune_history_path = os.path.join(history_dir, 'resnet50_finetuned_history.json')
history_resnet_head = load_history(head_history_path) if os.path.exists(head_history_path) else {}

# If history_resnet_finetune using
if 'history_resnet_finetune' not in locals() or not history_resnet_finetune:
    if os.path.exists(finetune_history_path):
        history_resnet_finetune = load_history(finetune_history_path)
    else:
        history_resnet_finetune = {}

if history_resnet_finetune:
    plot_training_curves(
        train_losses=history_resnet_finetune.get('train_loss', []),
        val_losses=history_resnet_finetune.get('val_loss', []),
        train_accs=history_resnet_finetune.get('train_acc', history_resnet_finetune.get('train_accuracy', [])),
        val_accs=history_resnet_finetune.get('val_acc', history_resnet_finetune.get('val_accuracy', [])),
        title="ResNet50 Fine-Tuning",
        save_path=os.path.join(history_dir, 'resnet50_finetune_curves.png')
    )

# Head + Fine-tune
if history_resnet_head and history_resnet_finetune:
    full_train_loss = history_resnet_head.get('train_loss', []) + history_resnet_finetune.get('train_loss', [])
    full_val_loss = history_resnet_head.get('val_loss', []) + history_resnet_finetune.get('val_loss', [])

    # Ստուգում ենք բանալու անունը (acc թե accuracy)
    head_train_acc = history_resnet_head.get('train_acc', history_resnet_head.get('train_accuracy', []))
    fine_train_acc = history_resnet_finetune.get('train_acc', history_resnet_finetune.get('train_accuracy', []))
    full_train_acc = head_train_acc + fine_train_acc

    head_val_acc = history_resnet_head.get('val_acc', history_resnet_head.get('val_accuracy', []))
    fine_val_acc = history_resnet_finetune.get('val_acc', history_resnet_finetune.get('val_accuracy', []))
    full_val_acc = head_val_acc + fine_val_acc

    # full curves
    plot_training_curves(
        train_losses=full_train_loss,
        val_losses=full_val_loss,
        train_accs=full_train_acc,
        val_accs=full_val_acc,
        title="ResNet50 Full Training (Head + Fine-Tune)",
        save_path=os.path.join(history_dir, 'resnet50_full_curves.png')
    )

# EfficentNet

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
CHECKPOINT_DIR = "/content/drive/MyDrive/Cassava_Project/checkpoints"

train_loader, valid_loader = get_dataloaders(
    train_df=train_df,
    val_df=val_df,
    img_dir=IMG_DIR,
    img_size=384,
    batch_size=16,    # 384x384 չափսի համար 16-ը հարմար է հիշողության (VRAM) համար
    num_workers=2
)
print(" EfficientNetV2-S Դատալոդերները պատրաստ են (384x384)!")

#  Հաշվարկում ենք դասերի կշիռները ճիշտ ֆունկցիայով
class_weights = calculate_class_weights(train_df, label_col='label').to(device)
print("Class weights:", class_weights)

# 5. Մոդելի, Loss-ի (կշռված), Optimizer-ի և Scheduler-ի ինիցիալիզացիա
model_name = "EfficientNetV2-S"
model = EfficientNetV2SCassava(num_classes=5, freeze_backbone=False).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

checkpoint_path = os.path.join(CHECKPOINT_DIR, "efficientnet_v2_s_head_best.pth")

if os.path.exists(checkpoint_path):
    print(f" {model_name}-ի Checkpoint-ը գտնվեց։ Բեռնում ենք քաշերը...")
    state_dict = torch.load(checkpoint_path, map_location=device)

    # Շտկում ենք բանալիների անհամապատասխանությունը
    new_state_dict = {}
    for k, v in state_dict.items():
        if not k.startswith("net."):
            k = f"net.{k}"
        if "classifier.1.weight" in k:
            k = k.replace("classifier.1.weight", "classifier.1.1.weight")
        elif "classifier.1.bias" in k:
            k = k.replace("classifier.1.bias", "classifier.1.1.bias")
        new_state_dict[k] = v

    model.load_state_dict(new_state_dict, strict=True)
    model.eval()
    print(" Մոդելը հաջողությամբ բեռնվեց:")
else:
    print(f" Checkpoint-ը չգտնվեց։ Սկսում ենք {model_name}-ի վարժեցումը...")
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=valid_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=5,
        scheduler=scheduler,
        save_path=checkpoint_path,
        use_amp=True,
        monitor="val_acc"
    )

verify_transfer_setup(model, model_name, device=device)

#  Վերջնական գնահատում և մատրիցի պահպանում
evaluate_model(
    model=model,
    val_loader=valid_loader,
    device=device,
    model_name=model_name,
    save_dir="/content/drive/MyDrive/Cassava_Project/plots"
)

# Effnet Fine tuned


In [ ]:


#  Մոդելի ինիցիալիզացիա
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
effnet_model = EfficientNetV2SCassava(num_classes=5, freeze_backbone=False).to(device)

# Ուղիների սահմանում
EFFNET_HEAD_PATH = '/content/drive/MyDrive/Cassava_Project/checkpoints/efficientnet_v2_s_head_best2.pth'
EFFNET_FINETUNE_PATH = '/content/drive/MyDrive/Cassava_Project/checkpoints/efficientnet_v2_s_finetuned_best.pth'

history_dir = '/content/drive/MyDrive/Cassava_Project/history'
finetune_history_path = os.path.join(history_dir, 'efficientnet_v2_s_finetuned_history.json')
full_history_path = os.path.join(history_dir, 'efficientnet_v2_s_full_history.json')

os.makedirs(os.path.dirname(EFFNET_FINETUNE_PATH), exist_ok=True)
os.makedirs(history_dir, exist_ok=True)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

#Checking checkpoint and doing fine-tuning

if os.path.exists(EFFNET_FINETUNE_PATH):
    print(f"Fine-tuned EfficientNetV2-S չեքփոինթը գտնվեց։ Բեռնում ենք քաշերը...")

    state_dict = torch.load(EFFNET_FINETUNE_PATH, map_location=device)

    new_state_dict = {}
    for k, v in state_dict.items():
        if not k.startswith("net."):
            k = f"net.{k}"
        new_state_dict[k] = v

    effnet_model.load_state_dict(new_state_dict)
    effnet_model.eval()
    print("Fine-tuned EfficientNetV2-S մոդելը հաջողությամբ բեռնվեց:")

    if os.path.exists(finetune_history_path):
        history_effnet_finetune = load_history(finetune_history_path)
        print(" Fine-tune history-ն հաջողությամբ բեռնվեց JSON-ից:")
    else:
        print("Fine-tune history JSON ֆայլը չգտնվեց:")
        history_effnet_finetune = {}

else:
    print("Fine-tuned չեքփոինթը չգտնվեց։ Սկսում ենք Fine-Tuning-ը Head Checkpoint-ի հիման վրա...")

    # 1. Բեռնում ենք Head-only չեքփոինթը
    if os.path.exists(EFFNET_HEAD_PATH):
        print(f" Բեռնում ենք Head-only չեքփոինթը՝ {EFFNET_HEAD_PATH}")
        head_state_dict = torch.load(EFFNET_HEAD_PATH, map_location=device)

        new_state_dict = {}
        for k, v in head_state_dict.items():
            if not k.startswith("net."):
                k = f"net.{k}"
            if "classifier.1.weight" in k and "classifier.1.1.weight" not in k:
                k = k.replace("classifier.1.weight", "classifier.1.1.weight")
            elif "classifier.1.bias" in k and "classifier.1.1.bias" not in k:
                k = k.replace("classifier.1.bias", "classifier.1.1.bias")
            new_state_dict[k] = v

        effnet_model.load_state_dict(new_state_dict, strict=True)
        print("Head-only կշիռները հաջողությամբ բեռնվեցին մոդելի մեջ:")
    else:
        print(" Head-only չեքփոինթը չգտնվեց, Fine-tuning-ը կսկսվի ընթացիկ մոդելից:")

    #  Ապասառեցնում ենք շերտերը fine-tuning-ի համար
    effnet_model = unfreeze_last_block(effnet_model)

    #  Optimizers
    finetune_optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, effnet_model.parameters()),
        lr=1e-5,
        weight_decay=1e-4
    )

    print("\n STARTING FINE-TUNING STAGE ")

    # 4. Training
    history_effnet_finetune = train_model(
        model=effnet_model,
        train_loader=train_loader,
        val_loader=valid_loader,
        criterion=criterion,
        optimizer=finetune_optimizer,
        device=device,
        epochs=5,
        save_path=EFFNET_FINETUNE_PATH,
        use_amp=True,
        monitor="val_acc"
    )

    #  ԱՅՍՏԵՂ ՊԱՀՈՒՄ ԵՆՔ FINE-TUNED .PTH-Ը DRIVE-ՈՒՄ
    torch.save(effnet_model.state_dict(), EFFNET_FINETUNE_PATH)
    print(f" Fine-tuned մոդելի կշիռները պահպանվեցին Drive-ում: {EFFNET_FINETUNE_PATH}")

    # 5. History Saving
    save_history_json(history_effnet_finetune, finetune_history_path)

    effnet_full_history = {
        "finetuned": history_effnet_finetune
    }
    save_history_json(effnet_full_history, full_history_path)
    print("Fine-tuning-ի history-ն պահպանվեց JSON-ում:")

In [ ]:


EFFNET_PATH = '/content/drive/MyDrive/Cassava_Project/checkpoints/efficientnet_v2_s_head_best2.pth'
EFFNET_FINETUNED_PATH = '/content/drive/MyDrive/Cassava_Project/checkpoints/efficientnet_v2_s_finetuned_best.pth'

history_dir = '/content/drive/MyDrive/Cassava_Project/history'
head_history_path = os.path.join(history_dir, 'efficientnet_v2_s_head_history.json')
finetune_history_path = os.path.join(history_dir, 'efficientnet_v2_s_finetuned_history.json')
full_history_path = os.path.join(history_dir, 'efficientnet_v2_s_full_history.json')

os.makedirs(history_dir, exist_ok=True)
os.makedirs(os.path.dirname(EFFNET_FINETUNED_PATH), exist_ok=True)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

if os.path.exists(EFFNET_FINETUNED_PATH):
    print(" Fine-tuned EfficientNet checkpoint-ը գտնվեց։ Բեռնում ենք քաշերը...")
    checkpoint_state = torch.load(EFFNET_FINETUNED_PATH, map_location=device)
    effnet_model.load_state_dict(checkpoint_state)
    effnet_model.eval()
    print(" Fine-tuned EfficientNetV2-S մոդելը հաջողությամբ բեռնվեց:")

    if os.path.exists(finetune_history_path):
        history_effnet_finetune = load_history(finetune_history_path)
        print(" Fine-tune history-ն հաջողությամբ բեռնվեց ֆայլից:")
    else:
        print(" Fine-tune history JSON ֆայլը չգտնվեց:")
        history_effnet_finetune = {}
else:
    print(" Fine-tuned checkpoint-ը չգտնվեց։ Սկսում ենք fine-tuning-ը...")

    # 1. Ապասառեցնում ենք վերջին բլոկները (կամ ամբողջ backbone-ը փոքր LR-ով)
    target_net = effnet_model.net if hasattr(effnet_model, 'net') else effnet_model
    for param in target_net.parameters():
        param.requires_grad = True

    # 2. Ստեղծում ենք ֆայնթյունինգի օպտիմիզատորը (փոքր learning rate-ով)
    finetune_optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, effnet_model.parameters()),
        lr=1e-5,
        weight_decay=1e-4
    )

    # 3. Գործարկում ենք թրեյնինգը
    history_effnet_finetune = train_model(
        model=effnet_model,
        train_loader=train_loader,
        val_loader=valid_loader,
        criterion=criterion,
        optimizer=finetune_optimizer,
        device=device,
        epochs=5,
        save_path=EFFNET_FINETUNED_PATH,
        use_amp=True,
        monitor="val_acc"
    )

    # Պահպանում ենք առանձին fine-tune պատմությունը
    save_history_json(history_effnet_finetune, finetune_history_path)

# 4. Միավորում ենք Stage 1 (head) և Stage 2 (finetuned) պատմությունները
history_effnet_head = load_history(head_history_path) if os.path.exists(head_history_path) else {}

effnet_full_history = {
    "head_only": history_effnet_head,
    "finetuned": history_effnet_finetune
}

save_history_json(effnet_full_history, full_history_path)
print(" EfficientNetV2-S ամբողջական պատմությունը հաջողությամբ միավորվեց և պահպանվեց:")

In [ ]:


head_history_path = os.path.join(history_dir, 'efficientnet_v2_s_head_history.json')
finetune_history_path = os.path.join(history_dir, 'efficientnet_v2_s_finetuned_history.json')

history_effnet_head = load_history(head_history_path) if os.path.exists(head_history_path) else {}

# Ստուգում ենք փոփոխականի առկայությունը memory-ում կամ բեռնում ֆայլից
if 'history_effnet_finetune' not in locals() or not history_effnet_finetune:
    if os.path.exists(finetune_history_path):
        history_effnet_finetune = load_history(finetune_history_path)
    else:
        history_effnet_finetune = {}

# 1. Գրաֆիկ միայն Fine-Tuning փուլի համար
if history_effnet_finetune:
    plot_training_curves(
        train_losses=history_effnet_finetune.get('train_loss', []),
        val_losses=history_effnet_finetune.get('val_loss', []),
        train_accs=history_effnet_finetune.get('train_acc', history_effnet_finetune.get('train_accuracy', [])),
        val_accs=history_effnet_finetune.get('val_acc', history_effnet_finetune.get('val_accuracy', [])),
        title="EfficientNetV2-S Fine-Tuning",
        save_path=os.path.join(history_dir, 'efficientnet_v2_s_finetune_curves.png')
    )

# 2. Գրաֆիկ Head + Fine-Tune միացյալ փուլերի համար
if history_effnet_head and history_effnet_finetune:
    full_train_loss = history_effnet_head.get('train_loss', []) + history_effnet_finetune.get('train_loss', [])
    full_val_loss = history_effnet_head.get('val_loss', []) + history_effnet_finetune.get('val_loss', [])

    head_train_acc = history_effnet_head.get('train_acc', history_effnet_head.get('train_accuracy', []))
    fine_train_acc = history_effnet_finetune.get('train_acc', history_effnet_finetune.get('train_accuracy', []))
    full_train_acc = head_train_acc + fine_train_acc

    head_val_acc = history_effnet_head.get('val_acc', history_effnet_head.get('val_accuracy', []))
    fine_val_acc = history_effnet_finetune.get('val_acc', history_effnet_finetune.get('val_accuracy', []))
    full_val_acc = head_val_acc + fine_val_acc

    plot_training_curves(
        train_losses=full_train_loss,
        val_losses=full_val_loss,
        train_accs=full_train_acc,
        val_accs=full_val_acc,
        title="EfficientNetV2-S Full Training (Head + Fine-Tune)",
        save_path=os.path.join(history_dir, 'efficientnet_v2_s_full_curves.png')
    )

In [ ]:


results_df = run_cassava_evaluation()

eval_path = '/content/drive/MyDrive/Cassava_Project/ֆայլեր'
if eval_path not in sys.path:
    sys.path.insert(0, eval_path)
os.chdir(eval_path)

# Ներմուծում ենք ֆունկցիաները
from evaluate import (
    run_cassava_evaluation,
    plot_model_results,
    finalize_best_model,
    compute_predictions,
    plot_misclassified_images,
    plot_random_validation_samples
)

print("Ներմուծումն հաջողվեց։ Սկսում ենք գնահատումը...")

# Գործարկում ենք գնահատումը
results_df = run_cassava_evaluation()
print(results_df)

In [ ]:

# Գործարկում ենք գնահատման և մոդելների համեմատության հիմնական ֆունկցիան
results_df = run_cassava_evaluation()

#  Նկարում և պահպանում ենք մոդելների համեմատության գրաֆիկը
plot_model_results(results_df)

#  Ընտրում ենք լավագույն մոդելը, անցկացնում վերջնական ստուգումն ու պահպանում ֆայլերը
# (Նկատի ունեցիր, որ ֆունկցիային պետք է փոխանցել նաև մարզված մոդելների փոփոխականները)
finalize_best_model(
    results_df=results_df,
    baseline_model=baseline_model,
    resnet_model=resnet_model,
    effnet_model=effnet_model,
    dino_model=dino_model,
    valid_loader=valid_loader,
    dinov2_valid_loader=dinov2_valid_loader,
    class_names=class_names,
    device=device,
    num_classes=NUM_CLASSES
)

#  Հաշվում ենք կանխատեսումները սխալների վերլուծության և վիզուալիզացիայի համար
# (Քանի որ finalize_best_model-ում արդեն որոշվել է լավագույն մոդելը, այստեղ կրկին վերցնում ենք այն)
best_model_name = results_df.sort_values("macro_f1", ascending=False).index[0]
model_lookup = {
    "BaselineCNN": baseline_model,
    "ResNet50": resnet_model,
    "EfficientNetV2S": effnet_model,
    "DINOv2Cassava": dino_model,
}
loader_lookup = {
    "BaselineCNN": valid_loader,
    "ResNet50": valid_loader,
    "EfficientNetV2S": valid_loader,
    "DINOv2Cassava": dinov2_valid_loader,
}

current_best_model = model_lookup[best_model_name]
current_best_loader = loader_lookup[best_model_name]

y_true, y_pred = compute_predictions(current_best_model, current_best_loader, device)
RESULTS_DIR = '/content/drive/MyDrive/Cassava_Project/results'
IMAGES_DIR = 'cassava_data/train_images'

#  Նկարում և ցույց ենք տալիս սխալ դասակարգված նկարները
plot_misclassified_images(
    y_true=y_true,
    y_pred=y_pred,
    final_model_name=best_model_name,
    final_loader=current_best_loader,
    val_df=val_df,
    images_dir=IMAGES_DIR,
    results_dir=RESULTS_DIR
)

#  Վերլուծում ենք շփոթված դասերը (Confusion Matrix-ի հիման վրա)
# (Անհրաժեշտ է ստեղծել կամ փոխանցել cm-ը, ուստի հաշվում ենք այն տեղում)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
eval_results_dict = {"confusion_matrix": cm}
val_df_reset = val_df.reset_index(drop=True)

analyze_confusion_errors(
    eval_results=eval_results_dict,
    y_true=y_true,
    y_pred=y_pred,
    val_df_reset=val_df_reset,
    images_dir=IMAGES_DIR,
    results_dir=RESULTS_DIR
)

#  Վերջում ցուցադրում ենք պատահական վալիդացիոն նմուշներ
plot_random_validation_samples(
    y_true=y_true,
    y_pred=y_pred,
    val_df=val_df,
    images_dir=IMAGES_DIR,
    results_dir=RESULTS_DIR
)